# How to save and load objects

Save a trained inference object or posterior with `save()` and load it with the
`load()` class method of the same class.

```python
from sbi.inference import NPE, DirectPosterior

# ... run inference
posterior = inference.build_posterior()

inference.save("/path/to/my_inference.pt")
posterior.save("/path/to/my_posterior.pt")

inference = NPE.load("/path/to/my_inference.pt")
posterior = DirectPosterior.load("/path/to/my_posterior.pt")
```

The loaded inference object can continue training. `load()` warns if the file was
saved with another sbi version. Files are unpickled, so only load files that you trust.

> `posterior` objects that were saved under `sbi v0.22.0` or older cannot be loaded under `sbi v0.23.0` or newer.

## I trained a model on a GPU. Can I load it on a CPU?

Yes. Pass `map_location="cpu"` to `load()`. The tensors and the device state move to
the CPU. `sample()`, `log_prob()` and training then run on the CPU.

```python
inference = NPE.load("/path/to/my_inference.pt", map_location="cpu")
posterior = DirectPosterior.load("/path/to/my_posterior.pt", map_location="cpu")
```

## Can I save only the network weights?

Yes. Save the `state_dict()` of the trained network. To load it, build the same network
again and load the weights into it. The weights file does not depend on how sbi stores
its objects, and `torch.load(..., weights_only=True)` reads it without running pickled
code. You need the code that builds the network.

```python
import torch
from sbi.inference import DirectPosterior
from sbi.neural_nets import posterior_nn

density_estimator = inference.train()
torch.save(density_estimator.state_dict(), "/path/to/my_weights.pt")

# Later: build the same network from one row of parameters and simulations.
estimator = posterior_nn(model="maf")(theta[:1], x[:1])
estimator.load_state_dict(torch.load("/path/to/my_weights.pt", weights_only=True))
posterior = DirectPosterior(estimator, prior)
```

The weights include the z-scoring statistics, so any row of the right shape works, as
long as its values are finite. sbi warns that statistics from one row do not represent
your data; here that is expected. Pass the same builder arguments as for training, for
example `model`, `hidden_features` or the embedding net. `NPE(density_estimator="maf")`
trains the network that `posterior_nn(model="maf")` builds.

For the mixed estimators `"mnle"` and `"mnpe"`, also pass `num_categories_per_variable`:
one row cannot show how many categories each discrete variable has.

Weights trained on a GPU load on a CPU-only machine with
`torch.load("/path/to/my_weights.pt", weights_only=True, map_location="cpu")`.